<a href="https://colab.research.google.com/github/KiranKBobba/Fin_Risk_Analytics/blob/collab/IT720_Assignment_4_Kiran_Kumar_Bobba.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### IT 720: NLP, Assignment 4 - A Tiny Transformer Language Model  175 Points

When you submit the assignment, make sure that you add your name to the file name.
When you think you are done, shut down the kernel one last time, restart it, and run your code from start to finish. Sometimes when you do this, you will discover that you still have errors. Leave the output in each cell to allow the grader to see it. If there is a bug that you cannot resolve, leave the error message in the output cell so the grader can see it.

#### Assignment Overview

For this assignment you will build your own language model using a simplified version of the original Transformer described in the Vaswani et. al. paper 'Attention is All You Need'. You do not have enough compute at home to build a "Large Language Model". However, even on a laptop you can build a model based on the alphabet rather than on words. This will reduce the computational load since there are far fewer alphabetic tokens than there are word tokens.

This model will probably take several hours to train. I suggest writing and debugging it using one epoch with a subset of the text for that epoch, and then, when your code works, training it for at least 2 epochs with the full text.

You will make many decisions for this model. Some of those decisions will be impacted by how much compute you have available. So when you first build your model, start small with only 1 transformer block with 2 attention heads, for example. Train it for one epoch to see how much time that takes, and make further decisions from there.

You can experiment with the number of transformer blocks, number of heads per block, embedding dimensions, and more. But be careful because a large model could easily require too much training time.  My own solution model has fewer than 500k trainable parameters and yet it still requires more than 1.5 hours per epoch.

After you train the model, you will also need to write an output function to generate text.  Detailed instructions for everything are given below.  

The purpose of this assignment is not to obtain impressive English output. Rather, the purpose is twofold:
1. Learn how to build a language model using the transformer architecture.
2. Gain some experience on the quality impact of various techniques for text generaton.


#### Rubric for the numbered sections below where you must write your own code, 175 points total

- 10 points: Task 1, Read and Preprocess the raw data
- 10 points: Task 2, Encode and Decode       
-  5 points: Task 3, set Hyperparameters
- 15 points: Task 4, Create training data
-  5 points: Task 5, causal mask function
- 35 points: Task 6, define TransformerBlock class  
- 35 points: Task 7, build Tiny Character Model    
- 15 points: Task 8, Compile and Summarize model     
- 10 points: Task 9, Train the model with at least 2 epochs
- 20 points: Task 10, Return character ID to generate  
- 15 points: Task 11, generate output text from seed

175 points total

In [1]:
# If you write your code in keras/tensorflow, then this cell has all of the imports you will need.
# You can write a solution using other packages, including PyTorch, if you prefer.
# However, many of my comments assume you are using Keras/TensorFlow, and it may be more
# work for you if you use PyTorch because some of those comments will be irrelevant.

import numpy as np
import os
import re
import tensorflow as tf
from   tensorflow.keras import layers

print("TensorFlow version:", tf.__version__)
print("Keras version:",      getattr(tf.keras, "__version__", "bundled with TensorFlow"))


TensorFlow version: 2.20.0
Keras version: 3.13.2


In [2]:
# Colab-friendly file loading. Upload Dracula.txt when prompted if it is not
# already present in the Colab working directory.
from pathlib import Path

text_path = Path("/content/Dracula.txt")
if not text_path.exists():
    try:
        from google.colab import files
        uploaded = files.upload()
        if "Dracula.txt" not in uploaded:
            raise FileNotFoundError("Please upload the file with the name Dracula.txt")
        text_path = Path("/content/Dracula.txt")
    except ImportError:
        # This fallback is useful when running the notebook outside Colab.
        text_path = Path("Dracula.txt")

raw_text = text_path.read_text(encoding="utf-8-sig")
print("BEFORE PREPROCESSING:\n")
print(raw_text[:1000])

# Keep printable characters plus newlines/tabs, and standardize line endings.
text = raw_text.replace("\r\n", "\n").replace("\r", "\n")
text = "".join(ch for ch in text if ch.isprintable() or ch in "\n\t")

print("\nAFTER PREPROCESSING:\n")
print(text[:1000])
print(f"\nCharacters before: {len(raw_text):,}")
print(f"Characters after:  {len(text):,}")


Saving Dracula.txt to Dracula.txt
BEFORE PREPROCESSING:

How these papers have been placed in sequence will be made manifest in
the reading of them. All needless matters have been eliminated, so that
a history almost at variance with the possibilities of later-day belief
may stand forth as simple fact. There is throughout no statement of
past things wherein memory may err, for all the records chosen are
exactly contemporary, given from the standpoints and within the range
of knowledge of those who made them.




DRACULA




CHAPTER I

JONATHAN HARKER’S JOURNAL

(_Kept in shorthand._)


_3 May. Bistritz._--Left Munich at 8:35 P. M., on 1st May, arriving at
Vienna early next morning; should have arrived at 6:46, but train was an
hour late. Buda-Pesth seems a wonderful place, from the glimpse which I
got of it from the train and the little I could walk through the
streets. I feared to go very far from the station, as we had arrived
late and would start as near the correct time as possible

In [3]:
characters = sorted(set(text))
vocabSize = len(characters)

char_to_id = {character: index for index, character in enumerate(characters)}
id_to_char = {index: character for character, index in char_to_id.items()}

print(f"Number of unique characters: {vocabSize}")
print("Unique characters:", repr("".join(characters)))

def encode(string):
    """Convert a string to a NumPy array of character IDs."""
    return np.array([char_to_id[character] for character in string], dtype=np.int32)

def decode(ids):
    """Convert an iterable of character IDs back to a string."""
    return "".join(id_to_char[int(character_id)] for character_id in ids)

vampire_encoded = encode("vampire")
print("Encoded 'vampire':", vampire_encoded)
print("Decoded array:", decode(vampire_encoded))


Number of unique characters: 93
Unique characters: '\n !&()*,-.0123456789:;?ABCDEFGHIJKLMNOPQRSTUVWXYZ_abcdefghijklmnopqrstuvwxyz{}£àáâæèéëïôö‘’“”'
Encoded 'vampire': [71 50 62 65 58 67 54]
Decoded array: vampire


In [4]:
# These settings produce a model with well under 500,000 trainable parameters.
batchSize  = 128
dimModel   = 128
epochs     = 2
ffDim      = 256
maxLength  = 129
numHeads   = 4
numLayers  = 2
vocabSize  = len(characters)
windowSize = 128

print({
    "batchSize": batchSize, "dimModel": dimModel, "epochs": epochs,
    "ffDim": ffDim, "maxLength": maxLength, "numHeads": numHeads,
    "numLayers": numLayers, "vocabSize": vocabSize, "windowSize": windowSize
})


{'batchSize': 128, 'dimModel': 128, 'epochs': 2, 'ffDim': 256, 'maxLength': 129, 'numHeads': 4, 'numLayers': 2, 'vocabSize': 93, 'windowSize': 128}


In [5]:
encoded_text = encode(text)

# Build sliding input/target windows. A stride of 4 keeps training practical
# in Colab while still providing many overlapping examples from the novel.
sequence_stride = 4
all_windows = np.lib.stride_tricks.sliding_window_view(
    encoded_text, window_shape=windowSize + 1
)[::sequence_stride]

inputs = np.asarray(all_windows[:, :-1], dtype=np.int32)
targets = np.asarray(all_windows[:, 1:], dtype=np.int32)

print("Input and target shapes:", inputs.shape, targets.shape)
print("First input row: ", repr(decode(inputs[0])))
print("First target row:", repr(decode(targets[0])))

train_dataset = (
    tf.data.Dataset.from_tensor_slices((inputs, targets))
    .shuffle(min(len(inputs), 50_000), seed=42, reshuffle_each_iteration=True)
    .batch(batchSize)
    .prefetch(tf.data.AUTOTUNE)
)


Input and target shapes: (210630, 128) (210630, 128)
First input row:  'How these papers have been placed in sequence will be made manifest in\nthe reading of them. All needless matters have been elimi'
First target row: 'ow these papers have been placed in sequence will be made manifest in\nthe reading of them. All needless matters have been elimin'


In [6]:
def causal_attention_mask(batch_size, sequence_length):
    """Return a boolean mask that prevents attention to future positions."""
    mask = tf.linalg.band_part(
        tf.ones((sequence_length, sequence_length), dtype=tf.bool), -1, 0
    )
    mask = tf.reshape(mask, (1, sequence_length, sequence_length))
    return tf.tile(mask, [batch_size, 1, 1])

# Display a small example of the causal mask.
print(causal_attention_mask(1, 5)[0].numpy().astype(int))


[[1 0 0 0 0]
 [1 1 0 0 0]
 [1 1 1 0 0]
 [1 1 1 1 0]
 [1 1 1 1 1]]


In [7]:
class TransformerBlock(layers.Layer):
    def __init__(self, dim_model, num_heads, ff_dim, dropout_rate=0.1):
        super().__init__()
        # key_dim is the size of each head; all heads together equal dim_model.
        self.attention = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=dim_model // num_heads,
            dropout=dropout_rate,
        )
        self.feed_forward = tf.keras.Sequential([
            layers.Dense(ff_dim, activation="gelu"),
            layers.Dense(dim_model),
        ])
        self.normalization_1 = layers.LayerNormalization(epsilon=1e-6)
        self.normalization_2 = layers.LayerNormalization(epsilon=1e-6)
        self.dropout_1 = layers.Dropout(dropout_rate)
        self.dropout_2 = layers.Dropout(dropout_rate)

    def call(self, inputs, training=None):
        batch_size = tf.shape(inputs)[0]
        sequence_length = tf.shape(inputs)[1]
        mask = causal_attention_mask(batch_size, sequence_length)

        attention_output = self.attention(
            inputs, inputs, attention_mask=mask, training=training
        )
        attention_output = self.dropout_1(attention_output, training=training)
        normalized_1 = self.normalization_1(inputs + attention_output)

        feed_forward_output = self.feed_forward(normalized_1, training=training)
        feed_forward_output = self.dropout_2(feed_forward_output, training=training)
        return self.normalization_2(normalized_1 + feed_forward_output)


In [8]:
class TinyCharacterModel(tf.keras.Model):
    def __init__(
        self, vocab_size, max_length, dim_model, num_heads, ff_dim, num_layers
    ):
        super().__init__()
        self.token_embedding = layers.Embedding(vocab_size, dim_model)
        self.position_embedding = layers.Embedding(max_length, dim_model)
        self.transformer_blocks = [
            TransformerBlock(dim_model, num_heads, ff_dim)
            for _ in range(num_layers)
        ]
        self.final_normalization = layers.LayerNormalization(epsilon=1e-6)
        self.output_logits = layers.Dense(vocab_size)

    def call(self, token_ids, training=None):
        sequence_length = tf.shape(token_ids)[1]
        positions = tf.range(start=0, limit=sequence_length, delta=1)
        x = self.token_embedding(token_ids) + self.position_embedding(positions)
        for transformer_block in self.transformer_blocks:
            x = transformer_block(x, training=training)
        x = self.final_normalization(x)
        return self.output_logits(x)

tinyModel = TinyCharacterModel(
    vocab_size=vocabSize,
    max_length=maxLength,
    dim_model=dimModel,
    num_heads=numHeads,
    ff_dim=ffDim,
    num_layers=numLayers,
)


In [9]:
loss_function = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
optimizer = tf.keras.optimizers.Adam(learning_rate=3e-4)

tinyModel.compile(optimizer=optimizer, loss=loss_function)

# Calling the subclassed model once creates its weights before summary().
buildVar = tf.zeros((1, windowSize), dtype=tf.int32)
_ = tinyModel(buildVar)

tinyModel.summary()


Model: "tiny_character_model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (1, 128, 128)          │        11,904 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_1 (Embedding)         │ (128, 128)             │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block               │ ?                      │       132,480 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_1             │ ?                      │       132,480 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ layer_normalization_4           │ (1, 128, 128)          │           256 │
│ (LayerNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (1, 128, 93)           │        11,997 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 305,629 (1.17 MB)

 Trainable params: 305,629 (1.17 MB)

 Non-trainable params: 0 (0.00 B)

In [10]:
history = tinyModel.fit(train_dataset, epochs=epochs)

# Save trained weights so training does not need to be repeated during the
# same Colab session if later generation experiments are rerun.
tinyModel.save_weights("tiny_dracula_transformer.weights.h5")


Epoch 1/2
1646/1646 ━━━━━━━━━━━━━━━━━━━━ 69s 32ms/step - loss: 2.1581
Epoch 2/2
1646/1646 ━━━━━━━━━━━━━━━━━━━━ 40s 24ms/step - loss: 1.7475


In [11]:
def return_next_character_id(model, context_ids, temperature=0.8, top_k=20):
    """Sample one next-character ID using temperature and top-k sampling."""
    if temperature <= 0:
        raise ValueError("temperature must be greater than zero")
    if top_k < 1:
        raise ValueError("top_k must be at least one")

    context_ids = np.asarray(context_ids, dtype=np.int32)[-windowSize:]
    model_input = tf.expand_dims(context_ids, axis=0)
    next_logits = model(model_input, training=False)[0, -1, :]
    next_logits = next_logits / temperature

    k = min(top_k, vocabSize)
    top_values, top_indices = tf.math.top_k(next_logits, k=k)
    top_probabilities = tf.nn.softmax(top_values)
    log_probabilities = tf.expand_dims(tf.math.log(top_probabilities), axis=0)
    sampled_position = tf.random.categorical(log_probabilities, num_samples=1)[0, 0]
    return int(top_indices[sampled_position].numpy())


In [12]:
def generate_text(
    model, seed, characters_to_generate=300, temperature=0.8, top_k=20
):
    """Generate a continuation from seed text using the trained model."""
    unknown = sorted(set(seed) - set(char_to_id))
    if unknown:
        raise ValueError(f"Seed contains unknown characters: {unknown}")

    generated_ids = list(encode(seed))
    for _ in range(characters_to_generate):
        next_id = return_next_character_id(
            model,
            generated_ids,
            temperature=temperature,
            top_k=top_k,
        )
        generated_ids.append(next_id)
    return decode(generated_ids)

seed = "Then Count Dracula said to me:"
generated_output = generate_text(
    tinyModel,
    seed=seed,
    characters_to_generate=300,
    temperature=0.8,
    top_k=20,
)
print(generated_output)


Then Count Dracula said to me:--

“Ahow fear eas even her byounds he apped to her last the hopened then the dan
then more not the nerhe came me the stak at would through of
miding the that looked a it not the hole mating shopen on the fort the requick
all at though looked my reartach, her mall serchen and to one the bee
gaing by


### Colab execution notes

1. Upload this notebook to Google Colab and select a GPU runtime.
2. Run all cells. When Task 1 prompts you, upload `Dracula.txt`.
3. Training is configured for the required two epochs. Runtime depends on the selected Colab hardware.
4. Before submission, rename the notebook if needed, restart the runtime, run all cells from start to finish, and retain the outputs for grading.
